## 用公式计算参数量和模型文件字节数

In [27]:
V = 32000
d_model = 8192
d_ff = 28672
L = 80

#V = 32000
#d_model = 4096
#d_ff = 11008
#L = 32

nparams = V*d_model + L*(4*d_model*d_model+3*d_ff*d_model + 2*d_model + 64) + d_model + V*d_model 
nbytes = 2*V*d_model*2 + L*(2*d_model*d_model + 2*d_model*1024+3*d_ff*d_model + 2*d_model)*2 + L*64*4 + d_model*2
print(nparams, nbytes)

78371894272 137953316864


## 用Hugging Face的Transformers模块读参数

In [ ]:
from transformers import AutoModel

# 加载模型
model = AutoModel.from_pretrained("c:/llama2/llama-2-7b-hf")

# 计算模型参数总数
total_params = sum(p.numel() for p in model.parameters())

print(f'Total parameters: {total_params}')

i = 1;
for p in model.parameters():
    print( p.shape)
    i = i + 1

C:\Users\HP\.conda\envs\pytorch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 直接读参数

In [26]:
import zipfile
import pickle
from pathlib import Path
from typing import (IO, Any, Callable)
from dataclasses import dataclass
import numpy as np
from abc import ABCMeta, abstractmethod

NDArray: 'TypeAlias' = 'np.ndarray[Any, Any]'

DT_F16 = 'F16'
DT_F32 = 'F32'

DataType = 'F16' 

@dataclass
class LazyStorageKind:
    data_type: DataType

@dataclass
class LazyStorage:
    load: Callable[[int, int], NDArray]
    kind: LazyStorageKind
    description: str

class Tensor(metaclass=ABCMeta):
    data_type: DataType

    @abstractmethod
    def astype(self, data_type: DataType) -> 'Tensor': ...
    @abstractmethod
    def permute(self, n_head: int, n_kv_head: Optional[int] = None) -> 'Tensor': ...
    @abstractmethod
    def permute_part(self, n_part: int, n_head: int) -> 'UnquantizedTensor': ...
    @abstractmethod
    def part(self, n_part: int) -> 'UnquantizedTensor': ...

class UnquantizedTensor(Tensor):
    def __init__(self, ndarray: NDArray) -> None:
        assert isinstance(ndarray, np.ndarray)
        self.ndarray = ndarray
        self.data_type = NUMPY_TYPE_TO_DATA_TYPE[ndarray.dtype]

    def permute_part(self, n_part: int, n_head: int) -> 'UnquantizedTensor':
        r = self.ndarray.shape[0] // 3
        return UnquantizedTensor(permute(self.ndarray[r * n_part : r * n_part + r, ...], n_head))

    def part(self, n_part: int) -> 'UnquantizedTensor':
        r = self.ndarray.shape[0] // 3
        return UnquantizedTensor(self.ndarray[r * n_part : r * n_part + r, ...])

    def permute(self, n_head: int, n_kv_head: Optional[int] = None) -> 'UnquantizedTensor':
        return UnquantizedTensor(permute(self.ndarray, n_head, n_kv_head))

@dataclass
class LazyTensor:
    _load: Callable[[], Tensor]
    shape: List[int]
    data_type: DataType
    description: str

    def load(self) -> Tensor:
        ret = self._load()
        assert ret.data_type == self.data_type, (self.data_type, ret.data_type, self.description)
        return ret

    def astype(self, data_type: DataType) -> 'LazyTensor':
        def load() -> Tensor:
            return self.load().astype(data_type)
        return LazyTensor(load, self.shape, data_type, f'convert({data_type}) {self.description}')
            
class LazyUnpickler(pickle.Unpickler):
    def __init__(self, fp: IO[bytes], data_base_path: str, zip_file: zipfile.ZipFile):
        super().__init__(fp)
        self.data_base_path = data_base_path
        self.zip_file = zip_file

    def persistent_load(self, pid: Any) -> Any:
        assert pid[0] == 'storage'
        assert isinstance(pid[1], LazyStorageKind)
        data_type = pid[1].data_type
        filename_stem = pid[2]
        filename = self.data_base_path + '/' + filename_stem
        info = self.zip_file.getinfo(filename)

        def load(offset: int, elm_count: int) -> NDArray:
            dtype = np.dtype(np.float16) #DATA_TYPE_TO_NUMPY.get(data_type)
            if dtype is None:
                raise Exception("tensor stored in unsupported format")
            fp = self.zip_file.open(info)
            fp.seek(offset * dtype.itemsize)
            
            size = elm_count * dtype.itemsize        
            data = fp.read(size)
                        
            assert len(data) == size
            return np.frombuffer(data, dtype)
        description = f'storage data_type={data_type} path-in-zip={filename} path={self.zip_file.filename}'
        return LazyStorage(load=load, kind=pid[1], description=description)    
    
    # @staticmethod
    def lazy_rebuild_tensor_v2(storage: Any, storage_offset: Any, size: Any, stride: Any,
                               # pyright: ignore[reportSelfClsParameterName]
                               requires_grad: Any, backward_hooks: Any, metadata: Any = None) -> LazyTensor:
        assert isinstance(storage, LazyStorage)

        def load() -> UnquantizedTensor:
            elm_count = stride[0] * size[0]
            print(f"elm = {elm_count}")
            return UnquantizedTensor(storage.load(storage_offset, elm_count).reshape(size))
        description = f'pickled storage_offset={storage_offset} in {storage.description}'
        return LazyTensor(load, list(size), storage.kind.data_type, description)
    
    # @staticmethod
    def rebuild_from_type_v2(func, new_type, args, state):
        return func(*args)
    
    CLASSES: Dict[Any, Any] = {
       # ('torch._tensor', '_rebuild_from_type_v2'): rebuild_from_type_v2,
        ('torch._utils', '_rebuild_tensor_v2'): lazy_rebuild_tensor_v2,
        #('torch', 'BFloat16Storage'): LazyStorageKind(DT_BF16),
        ('torch', 'HalfStorage'): LazyStorageKind(DT_F16),
        ('torch', 'FloatStorage'): LazyStorageKind(DT_F32),
        #('torch', 'IntStorage'): LazyStorageKind(DT_I32),
        ('torch', 'Tensor'): LazyTensor,
    }

    def find_class(self, module: str, name: str) -> Any:
        if not module.startswith('torch'):
            return super().find_class(module, name)
        return self.CLASSES[(module, name)]

if __name__ == '__main__':
    path = Path("C:\\llama2\\llama-2-7b-hf\\pytorch_model-00001-of-00002.bin")

    fp = open(path, 'rb')

    zf = zipfile.ZipFile(fp)

    pickle_paths = [name for name in zf.namelist() if name.endswith('.pkl')]
    assert len(pickle_paths) == 1, pickle_paths

    pickle_fp = zf.open(pickle_paths[0], 'r')
    unpickler = LazyUnpickler(pickle_fp,
                              data_base_path=pickle_paths[0][:-4],
                              zip_file=zf)   

    model = unpickler.load()
    as_dict = dict(model.items())

    nParams = 0
    nBytes = 0
    count = 0
    for name, lazy_tensor in as_dict.items():
        print(f"{name}: shape={lazy_tensor.shape} type={lazy_tensor.data_type}")
        if len(lazy_tensor.shape) == 1:
            count = lazy_tensor.shape[0]
        else:
            count = lazy_tensor.shape[0] * lazy_tensor.shape[1]

        if lazy_tensor.data_type == DT_F32:
            nBytes += count * 4
        else: #DT_F16
            nBytes += count * 2

        nParams += count

    print(f'参数量={nParams}, 字节数={nBytes}')

model.embed_tokens.weight: shape=[32000, 4096] type=F16
model.layers.0.self_attn.q_proj.weight: shape=[4096, 4096] type=F16
model.layers.0.self_attn.k_proj.weight: shape=[4096, 4096] type=F16
model.layers.0.self_attn.v_proj.weight: shape=[4096, 4096] type=F16
model.layers.0.self_attn.o_proj.weight: shape=[4096, 4096] type=F16
model.layers.0.self_attn.rotary_emb.inv_freq: shape=[64] type=F32
model.layers.0.mlp.gate_proj.weight: shape=[11008, 4096] type=F16
model.layers.0.mlp.up_proj.weight: shape=[11008, 4096] type=F16
model.layers.0.mlp.down_proj.weight: shape=[4096, 11008] type=F16
model.layers.0.input_layernorm.weight: shape=[4096] type=F16
model.layers.0.post_attention_layernorm.weight: shape=[4096] type=F16
model.layers.1.self_attn.q_proj.weight: shape=[4096, 4096] type=F16
model.layers.1.self_attn.k_proj.weight: shape=[4096, 4096] type=F16
model.layers.1.self_attn.v_proj.weight: shape=[4096, 4096] type=F16
model.layers.1.self_attn.o_proj.weight: shape=[4096, 4096] type=F16
model.l

In [10]:
4988274176 + 1750143488

6738417664